# ChromaDB와 Text2SQL
* Day 16 핵심 개념을 **짧은 코드로 한 번씩** 확인합니다.
* 데이터: `rag_data/` 의 **학부 학칙**·**일반대학원 학칙** PDF, `table_data.csv`


---
* `day15` Conda 환경을 사용합니다.
* Chroma·PDF·웹검색·SQLite 관련 패키지를 설치합니다.

In [ ]:
!pip install langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters langgraph pypdf chromadb ddgs pandas

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
RAG_DIR = WORKDIR / 'rag_data'
PDF_PATHS = sorted(RAG_DIR.glob('*.pdf'))  # 학부 + 일반대학원
CSV_PATH = RAG_DIR / 'table_data.csv'
CHROMA_DIR = WORKDIR / 'chroma_regulations'  # 학부·대학원 함께 인덱싱

print('WORKDIR :', WORKDIR)
print('PDFs    :', [p.name for p in PDF_PATHS])
print('CSV     :', CSV_PATH.name)


---
## 1. ChromaDB 구축

| 선택지 | 특징 |
|--------|------|
| FAISS | 가볍고 빠르지만, 운영·영구저장 기능이 약함 |
| ChromaDB | 로컬 persist 가능, LangChain 연동이 편함 |
| Milvus / Pinecone | 대규모·클라우드 |

### PDF 로드
* 학부 / 일반대학원 PDF를 각각 읽고, 파일명 기준으로 `level` 메타데이터를 붙입니다.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader


def infer_level(filename: str) -> str:
    """파일명으로 학부 / 대학원을 구분합니다. (학부를 먼저 검사)"""
    if '학부' in filename:
        return '학부'
    if '대학원' in filename:
        return '대학원'
    return '기타'


pages = []
for pdf_path in PDF_PATHS:
    loaded = PyPDFLoader(str(pdf_path)).load()
    level = infer_level(pdf_path.name)
    for doc in loaded:
        doc.metadata['level'] = level
        doc.metadata['doc_type'] = '학칙'
        doc.metadata['source_file'] = pdf_path.name
    print(f'{pdf_path.name} → level={level}, pages={len(loaded)}')
    pages.extend(loaded)

print('총 페이지:', len(pages))
print(pages[0].metadata)
print(pages[0].page_content[:200])


### Chunk + overlap
* chunk가 너무 작으면 문맥이 끊기고, 너무 크면 주제가 섞입니다.
* `chunk_overlap`으로 경계에서 문맥이 끊기는 것을 줄입니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
chunks = splitter.split_documents(pages)
print('chunk 수:', len(chunks))

In [ ]:
print(chunks[0].page_content[:400])
print('--- metadata ---')
print(chunks[0].metadata)

### Metadata 태깅
* 벡터 유사도만 쓰면 학부·대학원 조항이 섞여 나올 수 있습니다.
* 로드 단계에서 붙인 `level`(`학부` / `대학원`)을 chunk에도 유지하고, 검색 시 필터에 사용합니다.


In [ ]:
# chunk로 나눈 뒤에도 level이 유지되는지 확인
from collections import Counter

level_counts = Counter(c.metadata.get('level') for c in chunks)
print('level별 chunk 수:', dict(level_counts))

for level in ['학부', '대학원']:
    sample = next(c for c in chunks if c.metadata.get('level') == level)
    print(f'[{level}]', sample.metadata)


### Embedding 모델
* 모델마다 벡터 공간이 달라, 나중에 바꾸면 **전체 재인덱싱**이 필요합니다.
* 오늘은 실습 편의를 위해 OpenAI `text-embedding-3-small` 을 사용합니다.
* 거리 함수는 텍스트 검색에서 흔한 **cosine** 으로 맞춥니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(
    model='text-embedding-3-small',
    api_key=OPENAI_API_KEY,
)

v = embedding.embed_query('석사 수업연한은 몇 년인가?')
print('차원 수:', len(v))

### Collection 저장 (persist)
* 한 번 인덱싱하면 프로그램을 껐다 켜도 불러와 씁니다.
* 학부·대학원 PDF를 함께 넣었으므로, 예전에 만든 `chroma_hakchik` 폴더가 있어도 **이 셀의 `chroma_regulations`** 를 사용합니다.
* 폴더가 이미 있으면 재임베딩하지 않고 로드합니다. (다시 짓고 싶으면 해당 폴더를 삭제)


In [ ]:
from langchain_chroma import Chroma

if not CHROMA_DIR.exists():
    print('Creating new Chroma store')
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding,
        persist_directory=str(CHROMA_DIR),
        collection_metadata={'hnsw:space': 'cosine'},
    )
else:
    print('Loading existing Chroma store')
    vectorstore = Chroma(
        persist_directory=str(CHROMA_DIR),
        embedding_function=embedding,
    )

### 유사도 검색 (top-k)
* top-k가 너무 작으면 근거가 빠지고, 너무 크면 노이즈가 늘어납니다.
* 처음엔 넉넉히 가져온 뒤 Relevance Checker로 걸러내는 구성이 무난합니다.

In [ ]:
QUERY = '석사학위과정의 수업연한은 얼마인가?'
docs = vectorstore.similarity_search(QUERY, k=3)

for i, d in enumerate(docs, 1):
    print(f'[{i}] page={d.metadata.get("page")} | {d.page_content[:180]}')
    print('---')

### Metadata 필터
* `level`로 후보를 먼저 좁힌 뒤, 그 안에서 유사도 순위를 매깁니다.
* 같은 질문이라도 학부 / 대학원 필터 결과가 달라지는지 비교합니다.


In [ ]:
QUERY = '수업연한은 얼마인가?'

print('=== 필터 없음 (학부·대학원 섞일 수 있음) ===')
mixed = vectorstore.similarity_search(QUERY, k=3)
for d in mixed:
    print(d.metadata.get('level'), '|', d.metadata.get('source_file'), '|', d.page_content[:100].replace('\n', ' '))
    print('---')

for level in ['학부', '대학원']:
    print(f'=== filter level={level} ===')
    filtered = vectorstore.similarity_search(QUERY, k=2, filter={'level': level})
    print('건수:', len(filtered))
    for d in filtered:
        print(d.metadata.get('level'), '|', d.page_content[:120].replace('\n', ' '))
        print('---')


---
## 2. Relevance Checker + 웹 검색 보강

벡터 유사도가 높다고 해서 질문에 답이 되는 문서는 아닙니다.

```
Retrieve → Relevance Check → (통과) Generate
                           → (실패) Web Search → Generate
```

### State 정의

In [ ]:
from typing import Literal

from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


class RagState(BaseModel):
    question: str
    level: Literal['학부', '대학원', ''] = ''  # 비우면 필터 없음
    docs: list[str] = Field(default_factory=list)
    relevant: bool = False
    answer: str = ''
    source: Literal['rag', 'web', ''] = ''


llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, api_key=OPENAI_API_KEY)


### Retrieve Node

In [ ]:
def retrieve_node(state: RagState) -> dict:
    kwargs = {'k': 4}
    if state.level:
        kwargs['filter'] = {'level': state.level}
    hits = vectorstore.similarity_search(state.question, **kwargs)
    docs = [
        f"[level={d.metadata.get('level')}] {d.page_content}"
        for d in hits
    ]
    print('retrieve filter:', state.level or '(없음)', '| hits:', len(docs))
    return {'docs': docs}


### Relevance Check Node
* LLM에게 "이 문서가 질문에 답이 되는가"를 물어 통과/탈락을 가릅니다. (정성적 방식)

In [ ]:
def relevance_node(state: RagState) -> dict:
    context = '\n\n'.join(state.docs)
    prompt = (
        f'질문: {state.question}\n\n'
        f'문서:\n{context}\n\n'
        '문서만으로 질문에 답할 수 있으면 YES, 아니면 NO만 출력하세요.'
    )
    verdict = llm.invoke(prompt).content.strip().upper()
    ok = verdict.startswith('YES')
    print('Relevance:', verdict, '→', ok)
    return {'relevant': ok}

### Generate / Web Search Node

In [ ]:
import json

from ddgs import DDGS
from ddgs.exceptions import DDGSException


def generate_node(state: RagState) -> dict:
    context = '\n\n'.join(state.docs)
    prompt = (
        '아래 문서를 근거로 질문에 두세 문장으로 답하세요.\n\n'
        f'문서:\n{context}\n\n질문: {state.question}'
    )
    return {'answer': llm.invoke(prompt).content, 'source': 'rag'}


def web_search_node(state: RagState) -> dict:
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(state.question, max_results=3))
    except DDGSException:
        results = []
    context = json.dumps(results, ensure_ascii=False) if results else '검색 결과 없음'
    prompt = (
        '내부 학칙에서 답을 찾지 못했습니다. 웹 검색 결과를 참고해 답하세요.\n'
        '확실하지 않으면 그 사실을 명시하세요.\n\n'
        f'검색결과:\n{context}\n\n질문: {state.question}'
    )
    return {'answer': llm.invoke(prompt).content, 'source': 'web'}

### 그래프 연결

In [ ]:
from langgraph.graph import END, START, StateGraph


def route_after_relevance(state: RagState):
    return 'generate' if state.relevant else 'web_search'


rag_wf = StateGraph(RagState)
rag_wf.add_node('retrieve', retrieve_node)
rag_wf.add_node('relevance', relevance_node)
rag_wf.add_node('generate', generate_node)
rag_wf.add_node('web_search', web_search_node)

rag_wf.add_edge(START, 'retrieve')
rag_wf.add_edge('retrieve', 'relevance')
rag_wf.add_conditional_edges('relevance', route_after_relevance)
rag_wf.add_edge('generate', END)
rag_wf.add_edge('web_search', END)

rag_app = rag_wf.compile()
rag_app

### 실행 — 학칙에 있을 법한 질문

In [ ]:
# 대학원 학칙만 보고 싶을 때 level='대학원'
out = rag_app.invoke(RagState(question='석사학위과정의 수업연한은 얼마인가요?', level='대학원'))
print('source:', out['source'])
print(out['answer'])


### 실행 — 학칙에 없을 법한 질문 (웹 검색으로 넘어가는지 확인)

In [ ]:
out = rag_app.invoke(RagState(question='2026년 최신 HBM4 스펙 요약해줘'))
print('source:', out['source'])
print(out['answer'])

---
## 3. Text2SQL

* "평균이 얼마야?" 같은 **숫자 계산**은 벡터 검색이 아니라 SQL이 맞습니다.
* `table_data.csv`(콘크리트 배합·강도)를 SQLite로 올려 SELECT만 실행합니다.

### CSV → SQLite

In [ ]:
import sqlite3

import pandas as pd

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]

DB_PATH = WORKDIR / 'concrete.db'
conn = sqlite3.connect(DB_PATH)
df.to_sql('concrete', conn, if_exists='replace', index=False)

print(df.shape)
print(df.columns.tolist())
df.head(3)

### 스키마 + Few-shot
* LLM은 DB 구조를 모르므로 **스키마**를 함께 줘야 합니다.
* "이런 질문 → 이런 SQL" 예시(Few-shot)를 넣으면 JOIN·집계 실수가 줄어듭니다.

In [ ]:
SCHEMA = """
Table: concrete
Columns:
- Cement (float): 시멘트량
- Blast_Furnace_Slag (float)
- Fly_Ash (float)
- Water (float): 물량
- Superplasticizer (float)
- Coarse_Aggregate (float)
- Fine_Aggregate (float)
- Age (int): 양생 일수
- Concrete_compressive_strength (float): 압축강도
"""

FEW_SHOT = """
Q: Age가 28일인 샘플의 평균 압축강도는?
SQL: SELECT AVG(Concrete_compressive_strength) FROM concrete WHERE Age = 28;

Q: Cement가 400 이상인 행은 몇 개인가?
SQL: SELECT COUNT(*) FROM concrete WHERE Cement >= 400;
"""

### SQL 생성
* 응답은 SQL문만 나오게 하고, 실행 전에 **SELECT만 허용**합니다. (안전장치)

In [ ]:
def question_to_sql(question: str) -> str:
    prompt = (
        'SQLite SELECT 쿼리만 한 줄로 작성하세요. 설명·코드펜스 금지.\n'
        f'{SCHEMA}\n예시:\n{FEW_SHOT}\n'
        f'질문: {question}\nSQL:'
    )
    sql = llm.invoke(prompt).content.strip().strip('`')
    if sql.lower().startswith('sql'):
        sql = sql[3:].strip()
    return sql


def run_select(sql: str):
    cleaned = sql.strip().rstrip(';')
    if not cleaned.lower().startswith('select'):
        return '오류: SELECT만 허용합니다.', sql
    banned = ('insert', 'update', 'delete', 'drop', 'alter', 'attach')
    low = cleaned.lower()
    if any(b in low for b in banned):
        return '오류: 위험 키워드가 포함되어 실행을 거부합니다.', sql
    try:
        cur = conn.execute(cleaned)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description] if cur.description else []
        return {'columns': cols, 'rows': rows}, sql
    except Exception as e:
        return f'결과 없음/실행 실패: {e}', sql

### 실행

In [ ]:
q = 'Age가 28일인 샘플의 평균 압축강도는 얼마인가?'
sql = question_to_sql(q)
result, sql = run_select(sql)

print('SQL:', sql)
print('결과:', result)

In [ ]:
q = 'Cement가 400 이상인 행은 몇 개인가?'
sql = question_to_sql(q)
result, sql = run_select(sql)

print('SQL:', sql)
print('결과:', result)

### 결과를 자연어로 한 번 더

In [ ]:
def sql_answer(question: str) -> str:
    sql = question_to_sql(question)
    result, sql = run_select(sql)
    prompt = (
        f'질문: {question}\n실행 SQL: {sql}\n결과: {result}\n'
        '위 결과를 한두 문장으로 한국어로 설명하세요.'
    )
    return llm.invoke(prompt).content


print(sql_answer('Age가 28일인 샘플의 평균 압축강도는 얼마인가?'))

---
## 4. 정리

| 질문 유형 | 도구 |
|-----------|------|
| 규정·절차·정의 | Chroma 검색 (+ Relevance) |
| 평균·건수·집계 | Text2SQL |
| 내부에 없음 | 웹 검색 |

In [ ]:
print('=== RAG (대학원 필터) ===')
r = rag_app.invoke(RagState(question='석사 수업연한을 알려줘', level='대학원'))
print(r['source'], '|', r['answer'][:200])

print('\n=== RAG (학부 필터) ===')
r2 = rag_app.invoke(RagState(question='학부 수업연한(졸업이수학기) 관련 규정을 알려줘', level='학부'))
print(r2['source'], '|', r2['answer'][:200])

print('\n=== Text2SQL ===')
print(sql_answer('Water가 200 이상인 샘플의 평균 압축강도는?'))
